# 02 · 在 CIFAR-10 上訓練一個 CNN

`ml/pytorch` 軌道你在灰階的 MNIST 上跑過 CNN。這課升級到**彩色、更難**的 CIFAR-10,並用上現代 CNN 的標準配備:**Conv → BatchNorm → ReLU** 堆疊、**MaxPool** 降尺寸、**Dropout** 抗過擬合。

目標不是衝 SOTA,而是讓你看到一個「像樣的」CNN 怎麼組、怎麼在真實彩色資料上學起來。

## 1. 安裝與資料

In [ ]:
%pip install -q -U torchvision

In [ ]:
# CIFAR-10 的 10 個類別，以及這個資料集慣用的正規化平均/標準差。
CIFAR10_CLASSES = ["plane", "car", "bird", "cat", "deer",
                   "dog", "frog", "horse", "ship", "truck"]
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)

In [ ]:
import torch
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
tf = transforms.Compose([transforms.ToTensor(),
                         transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD)])
train_set = torchvision.datasets.CIFAR10("./data", train=True, download=True, transform=tf)
test_set = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=tf)
train_loader = DataLoader(train_set, batch_size=128, shuffle=True, num_workers=2)
test_loader = DataLoader(test_set, batch_size=256, num_workers=2)
print(f"訓練 {len(train_set)} 張、測試 {len(test_set)} 張，device={device}")

## 2. 一個現代風格的小 CNN

三個「卷積區塊」,每區塊兩層 conv + BatchNorm + ReLU,再 MaxPool 把尺寸減半(32→16→8→4)。最後攤平接全連接層分類。

In [ ]:
import torch.nn as nn


def conv_block(cin, cout):
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        nn.MaxPool2d(2),
    )


class SmallCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.features = nn.Sequential(conv_block(3, 32), conv_block(32, 64), conv_block(64, 128))
        self.head = nn.Sequential(
            nn.Flatten(), nn.Linear(128 * 4 * 4, 256), nn.ReLU(inplace=True),
            nn.Dropout(0.3), nn.Linear(256, n_classes),
        )

    def forward(self, x):
        return self.head(self.features(x))


model = SmallCNN().to(device)
print(sum(p.numel() for p in model.parameters()), "個參數")

## 3. 訓練

標準訓練迴圈。CIFAR-10 約 5–8 個 epoch 就能看到明顯成效(T4 幾分鐘)。

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
EPOCHS = 6

for ep in range(1, EPOCHS + 1):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        opt.step()
    print(f"epoch {ep}/{EPOCHS}  最後一批 loss {loss.item():.3f}")
print("訓練完成。")

## 4. 評估準確率 + 看幾個預測

In [ ]:
model.eval()
correct = total = 0
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        pred = model(x).argmax(1)
        correct += (pred == y).sum().item()
        total += y.size(0)
print(f"測試準確率:{100 * correct / total:.1f}%")

## 小結

- 在彩色、較難的 CIFAR-10 上,組了一個有 **BatchNorm / Dropout** 的現代小 CNN。
- 訓練幾個 epoch 就能到不錯的準確率——但純手刻 CNN 要再往上很吃資料與算力。
- 下一課的**遷移學習**會讓你用更少資料、更快達到更高準確率。

下一課:不從零訓練,改站在 ImageNet 預訓練模型的肩上。